# Analysis and Draft Plotting Workflow

Use this notebook for graph loading, force-cluster extraction, exploratory statistics, loop analysis, and rough draft plots. Keep polished manuscript/presentation figures in `02_publication_figure_workflow.ipynb`.

This organized copy points at the current graph-output structure under `AnalysisResults/FinalLoadState`, `AnalysisResults/JammingState`, and `AnalysisResults/LoadStateComparison`. The original notebook is left unchanged in `post_analysis/`.

# Granular Project Analysis:
The input graphs generated from simulations are already calculated with graph properties and other distributes. Here the main task are as following:
- Extract the force clusters and get analysis for the force chains;
  - Type analysis
  - Size analysis
  - Density analysis - Done✅
  - Graph property analaysis - Done✅
  - Cluster subgraph property analsysis
  - Customized betweenness/closeness?
  - Force chain direction analysis
  - Force chain direction change analysis
- Get the displacement and stress relations
  - stress vs. displacement (bin plots/trending, etc.)
  - force chain vs. displacement
- Perform further analysis with loops in the graphs (loop type/numbers)
- Perform further analysis with perculation in the graphs
- Plotting session (3D graph representation) - Done✅


## 0) Load in needed packages

In [ ]:
import os
import pickle
import random
import math
import itertools
import numpy as np
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from matplotlib.colors import to_rgb, Normalize, ListedColormap
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.cm import ScalarMappable
import matplotlib.cm as cm
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors

## 1) Load in the graph dataset and set output path

In [ ]:
# ---------- Current graph output locations ----------
from pathlib import Path

ROOT = Path("/scratch/abucsek_root/abucsek0/yfjin/Granular_RubySim")
POST_ANALYSIS_DIR = ROOT / "AnalysisScripts/post_analysis"
FINAL_GRAPH_DIR = ROOT / "AnalysisResults/FinalLoadState/FullGraph_2mean_ref_geom"
JAMMING_ORIGINAL_GRAPH_DIR = ROOT / "AnalysisResults/JammingState/FullGraph_2mean_ref_geom"
JAMMING_FINAL_THRESHOLD_GRAPH_DIR = ROOT / "AnalysisResults/JammingState/FullGraph_final_load_threshold"
COMPARE_DIR = ROOT / "AnalysisResults/LoadStateComparison/Final_vs_Jamming_final_threshold"

# Choose which graph list this notebook should analyze.
# Options: "final", "jamming_original", "jamming_final_threshold"
GRAPH_STATE = "final"
GRAPH_DIRS = {
    "final": FINAL_GRAPH_DIR,
    "jamming_original": JAMMING_ORIGINAL_GRAPH_DIR,
    "jamming_final_threshold": JAMMING_FINAL_THRESHOLD_GRAPH_DIR,
}
GRAPH_DIR = GRAPH_DIRS[GRAPH_STATE]
GRAPH_LABELED_PATH = GRAPH_DIR / "graph_dict_labeled.pkl"
GRAPH_WITH_CLUSTERS_PATH = GRAPH_DIR / "graph_dict_labeled_with_clusters.pkl"

with open(GRAPH_LABELED_PATH, "rb") as f:
    GL_final = pickle.load(f)

# Keep legacy variable names used later in the notebook.
GL = GL_final
out_dir = str(GRAPH_DIR)
print(f"Loaded {GRAPH_STATE} graph_dict from: {GRAPH_LABELED_PATH}")
print("Geometries:", list(GL_final.keys()))

## 2) Extract the force clusters and save the cluster information

### 2.1) Set parameters for defining force clusters and exact files for saving

In [ ]:
# --------- Settings per request -----------
GRAPH_VIEW = 'full'          # include walls & wall contacts
MIN_NODES_PER_CLUSTER = 3
MIN_EDGES_PER_CLUSTER = 2

tag = f"{GRAPH_VIEW}_minN{MIN_NODES_PER_CLUSTER}_E{MIN_EDGES_PER_CLUSTER}"
CLUSTERS_CSV = os.path.join(out_dir, f"high_force_clusters_{tag}.csv")
CLUSTER_EDGES_CSV = os.path.join(out_dir, f"high_force_cluster_edges_{tag}.csv")
CLUSTER_NODES_CSV = os.path.join(out_dir, f"high_force_cluster_nodes_{tag}.csv")
OUT_PICKLE = os.path.join(out_dir, f"graph_dict_with_clusters_{tag}.pkl")

### 2.2) Helper function to extract force clusters

In [ ]:
# label high-force clusters from a single graph
def label_high_force_clusters_full(G):
    """
    Label connected clusters of high-force contacts on a *full* graph G.

    Writes on nodes/edges in a cluster:
      - 'hf_cluster_id'   : int (1..K), ranked by cluster size (#nodes desc, then #edges desc)
      - 'hf_cluster_type' : {-100, 100, 1}
        * -100: exactly 2 nodes and 1 edge (a single HF contact)
        *  100: exactly 3 nodes and at least 2 wall nodes (wall–particle–wall)
        *    1: all other clusters

    Returns
    -------
    clusters : list[dict] with keys: id, type, graph
    """
    if G.is_multigraph():
        hf_edges = [
            (u, v, k)
            for u, v, k, data in G.edges(keys=True, data=True)
            if data.get("is_high_force", False)
        ]
    else:
        hf_edges = [
            (u, v)
            for u, v, data in G.edges(data=True)
            if data.get("is_high_force", False)
        ]

    if not hf_edges:
        return []

    H = G.edge_subgraph(hf_edges).copy()
    components = list(nx.connected_components(H))

    cluster_records = []
    for comp_nodes in components:
        compH = H.subgraph(comp_nodes).copy()
        n_nodes = compH.number_of_nodes()
        n_edges = compH.number_of_edges()
        n_wall = sum(1 for n in compH.nodes if G.nodes[n].get("is_wall", False))

        if n_nodes == 2 and n_edges == 1:
            ctype = -100
        elif n_nodes == 3 and n_wall >= 2:
            ctype = 100
        else:
            ctype = 1

        cluster_records.append({
            "graph": compH,  # just the subgraph
            "type": ctype,
        })

    # sort by size for consistent ID assignment
    cluster_records.sort(key=lambda d: (d["graph"].number_of_nodes(),
                                        d["graph"].number_of_edges()), reverse=True)

    for cid, rec in enumerate(cluster_records, start=1):
        rec["id"] = cid
        ctype = rec["type"]
        for n in rec["graph"].nodes:
            G.nodes[n]["hf_cluster_id"] = cid
            G.nodes[n]["hf_cluster_type"] = ctype
        if G.is_multigraph():
            for u, v, k in rec["graph"].edges(keys=True):
                G[u][v][k]["hf_cluster_id"] = cid
                G[u][v][k]["hf_cluster_type"] = ctype
        else:
            for u, v in rec["graph"].edges():
                G[u][v]["hf_cluster_id"] = cid
                G[u][v]["hf_cluster_type"] = ctype

    return cluster_records

# go through all graphs
def label_all_full_graphs(GL_final):
    """
    Run high-force cluster labeling for all graphs.
    Returns
    -------
    summary : dict[angle] -> list of per-simulation summaries
    clusters_index : dict[angle] -> list of cluster lists (per sim),
                     where each cluster is a dict with keys:
                        id, type, graph
    """
    summary = {}
    clusters_index = {}
    for angle, d in GL_final.items():
        angle_summary = []
        angle_clusters = []
        for idx, G_full in enumerate(d["full"]):
            clusters = label_high_force_clusters_full(G_full)
            # summary info only
            angle_summary.append({
                "sim_idx": idx,
                "n_clusters": len(clusters),
                "types": [c["type"] for c in clusters],
                "sizes": [c["graph"].number_of_nodes() for c in clusters],
            })
            # keep the actual cluster records (with subgraphs)
            angle_clusters.append(clusters)
        summary[angle] = angle_summary
        clusters_index[angle] = angle_clusters
    return summary, clusters_index

# Save the extracted cluster information
def save_updated_graphs(GL_final, clusters_index, out_dir, suffix="_with_clusters"):
    """Save GL_final and cluster index under new filenames."""
    new_file = os.path.join(out_dir, f"graph_dict_labeled{suffix}.pkl")
    with open(new_file, "wb") as f:
        pickle.dump(GL_final, f)
    print(f"✅ Updated graphs saved to: {new_file}")

    clusters_file = os.path.join(out_dir, f"clusters_index.pkl")
    with open(clusters_file, "wb") as f:
        pickle.dump(clusters_index, f)
    print(f"✅ Cluster index saved to: {clusters_file}")

    return new_file, clusters_file

### 2.3) Run and do the extraction

In [ ]:
summary, clusters_index = label_all_full_graphs(GL_final)
updated_path, clusters_path = save_updated_graphs(GL_final, clusters_index, out_dir)

# Quick peek: first cluster of first sim at 0°
c0 = clusters_index["0deg"][0][0]
print(c0["id"], c0["type"], c0["graph"].nodes(), c0["graph"].edges())

## 3) Force cluster analysis

### 3.1) Graph property (full graph) of force cluster vs. no force cluster

#### 3.1.1) Load in saved graphs with cluster

In [ ]:
# Load clustered graph data when it exists; otherwise use the labeled graph.
# The corrected jamming folder does not carry old cluster files, because changing
# the high-force threshold invalidates old cluster membership.
cluster_path = GRAPH_WITH_CLUSTERS_PATH if GRAPH_WITH_CLUSTERS_PATH.exists() else GRAPH_LABELED_PATH
with open(cluster_path, "rb") as f:
    GL = pickle.load(f)

clusters_index_path = GRAPH_DIR / "clusters_index.pkl"
if clusters_index_path.exists():
    with open(clusters_index_path, "rb") as f:
        clusters_index = pickle.load(f)
else:
    clusters_index = None

print(f"Loaded graph data from: {cluster_path}")
print("Loaded geometries:", list(GL.keys()))
print("clusters_index:", "loaded" if clusters_index is not None else "not present for this graph folder")

#### 3.1.2) Feature to extract

In [ ]:
NODE_FEATURES = [
    "stress_vm",
    "stress_hydro",
    "degree",
    "closeness",
    "betweenness",
    "clustering",
    "avg_neighbor_degree",
    "principal_eigenvector",
    "fiedler",
    "nfd",
    "nfd_r2",
    "degree_with_walls",
    "closeness_with_walls",
    "betweenness_with_walls",
    "clustering_with_walls",
    "avg_neighbor_degree_with_walls",
    "principal_eigenvector_with_walls",
    "fiedler_with_walls",
    "high_force_degree",
]

EDGE_FEATURES = [
    "normal_force",
    "tangential_force",
    "delta",
    "delta_t",
    "angle_with_zz",
    "curvature_with_walls",
    "curvature_no_walls",
    "edge_connectivity",
    "edge_connectivity_with_walls",
    "node_connectivity",
    "node_connectivity_with_walls",
]

#### 3.1.3) Group the properties

In [ ]:
node_stats = {}  # node_stats[geom]["cluster"/"noncluster"] = list of dict rows
edge_stats = {}  # edge_stats[geom]["cluster"/"noncluster"] = list of dict rows


def safe_extract(attr_dict, key):
    """Return attr value or np.nan if missing."""
    return attr_dict[key] if key in attr_dict else np.nan


for geom, geom_dict in GL.items():
    
    print(f"Processing geometry: {geom}")

    node_stats[geom] = {"cluster": [], "noncluster": []}
    edge_stats[geom] = {"cluster": [], "noncluster": []}

    # Use FULL graphs (clusters & hf_cluster_type live here,
    # and core metrics were copied back)
    for G in geom_dict["full"]:

        # ---------- NODE PROCESSING ----------
        for n, data in G.nodes(data=True):
            row = {}

            # wall flag for later "no-wall" filtering
            row["_is_wall"] = bool(data.get("is_wall", False))

            # Only treat TYPE-1 clusters as "cluster"
            # Nodes in type -100 or 100 or with no label -> "noncluster"
            ctype = data.get("hf_cluster_type", None)
            in_type1_cluster = (ctype == 1)

            for feat in NODE_FEATURES:
                row[feat] = safe_extract(data, feat)

            if in_type1_cluster:
                node_stats[geom]["cluster"].append(row)
            else:
                node_stats[geom]["noncluster"].append(row)

        # ---------- EDGE PROCESSING ----------
        # Graph is simple Graph, so we can use edges(data=True)
        edge_iter = G.edges(data=True)
        
        for u, v, data in edge_iter:

            row = {}

            # Only TYPE-1 cluster edges counted as "cluster"
            ctype = data.get("hf_cluster_type", None)
            in_type1_cluster = (ctype == 1)

            for feat in EDGE_FEATURES:
                row[feat] = safe_extract(data, feat)

            if in_type1_cluster:
                edge_stats[geom]["cluster"].append(row)
            else:
                edge_stats[geom]["noncluster"].append(row)


#### 3.1.4) Means with wall nodes

In [ ]:
def compute_mean_std(rows, features):
    """Return dict: feature -> (mean, stderr)."""
    if len(rows) == 0:
        return {f: (np.nan, np.nan) for f in features}
    out = {}
    for f in features:
        vals = np.array([r[f] for r in rows], dtype=float)
        vals = vals[~np.isnan(vals)]
        if len(vals) == 0:
            out[f] = (np.nan, np.nan)
        else:
            mean = np.mean(vals)
            stderr = np.std(vals, ddof=1) / np.sqrt(len(vals))
            out[f] = (mean, stderr)
    return out


node_stats_mean = {}
edge_stats_mean = {}

for geom in GL.keys():
    node_stats_mean[geom] = {
        "cluster":    compute_mean_std(node_stats[geom]["cluster"], NODE_FEATURES),
        "noncluster": compute_mean_std(node_stats[geom]["noncluster"], NODE_FEATURES),
    }
    edge_stats_mean[geom] = {
        "cluster":    compute_mean_std(edge_stats[geom]["cluster"], EDGE_FEATURES),
        "noncluster": compute_mean_std(edge_stats[geom]["noncluster"], EDGE_FEATURES),
    }

#### 3.1.5) Means without wall nodes

In [ ]:
def filter_out_walls(rows):
    """Return only rows where _is_wall is False."""
    return [r for r in rows if not r.get("_is_wall", False)]


node_stats_mean_no_wall = {}

for geom in GL.keys():
    node_stats_mean_no_wall[geom] = {
        "cluster": compute_mean_std(
            filter_out_walls(node_stats[geom]["cluster"]), NODE_FEATURES
        ),
        "noncluster": compute_mean_std(
            filter_out_walls(node_stats[geom]["noncluster"]), NODE_FEATURES
        ),
    }

#### 3.1.6) Plot the mean of two groups

In [ ]:
# Geometry ordering & labels for plotting
geom_keys   = ["0deg", "15deg", "30deg", "45deg"]
geom_labels = ["0°",   "15°",   "30°",   "45°"]

# With walls
node_cluster_means        = {g: node_stats_mean[g]["cluster"]    for g in geom_keys}
node_noncluster_means     = {g: node_stats_mean[g]["noncluster"] for g in geom_keys}
edge_cluster_means        = {g: edge_stats_mean[g]["cluster"]    for g in geom_keys}
edge_noncluster_means     = {g: edge_stats_mean[g]["noncluster"] for g in geom_keys}

# Without wall particles (nodes only)
node_cluster_means_nw     = {g: node_stats_mean_no_wall[g]["cluster"]    for g in geom_keys}
node_noncluster_means_nw  = {g: node_stats_mean_no_wall[g]["noncluster"] for g in geom_keys}


# ============================================================
# 7. PLOTTING FUNCTION (CLUSTER vs NON-CLUSTER)
# ============================================================

# Global Matplotlib style
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 16
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 18
plt.rcParams["xtick.labelsize"] = 16
plt.rcParams["ytick.labelsize"] = 16

# Colors by geometry (purple, blue, green, gold)
palette     = ["#7B68EE", "#1E90FF", "#3CB371", "#FFD700"]


def plot_feature_list_compare(
    feature_list,
    cluster_dict,
    noncluster_dict,
    title_prefix="",
    save=False,
    save_dir=None,
    file_suffix="",
    dpi=200,
    x_offset=0.0,   # 0.0 = exact same x; >0 = slight horizontal separation
):
    """
    Make scatter+error bar plots comparing cluster vs non-cluster on SAME plot.

    cluster_dict / noncluster_dict:
        {geom_key: {feature: (mean, stderr), ...}, ...}
    """

    if save:
        if save_dir is None:
            raise ValueError("If save=True, you must specify save_dir.")
        os.makedirs(save_dir, exist_ok=True)

    xs = np.arange(len(geom_keys))

    for feature in feature_list:
        fig, ax = plt.subplots(figsize=(6.5, 4.5))

        for i, g in enumerate(geom_keys):
            mean_c, err_c   = cluster_dict[g][feature]
            mean_nc, err_nc = noncluster_dict[g][feature]

            # Cluster: circle
            ax.errorbar(
                xs[i] - x_offset, mean_c,
                yerr=err_c,
                fmt="o",
                markersize=8,
                capsize=3,
                color=palette[i],
                ecolor=palette[i],
                elinewidth=1.2,
            )

            # Non-cluster: square
            ax.errorbar(
                xs[i] + x_offset, mean_nc,
                yerr=err_nc,
                fmt="s",
                markersize=7,
                capsize=3,
                color=palette[i],
                ecolor=palette[i],
                elinewidth=1.2,
            )

        ax.set_xticks(xs)
        ax.set_xticklabels(geom_labels)
        ax.set_ylabel(f"Mean {feature}")
        ax.set_title(f"{title_prefix}{feature}")

        # No grid, no legend
        ax.grid(False)

        plt.tight_layout()

        if save:
            fname = f"{feature}{file_suffix}.png"
            fpath = os.path.join(save_dir, fname)
            fig.savefig(fpath, dpi=dpi)
            print(f"Saved: {fpath}")

        plt.show()


# ============================================================
# 8. EXAMPLE CALLS
# ============================================================

# A) Node features, INCLUDING wall particles
plot_feature_list_compare(
    NODE_FEATURES,
    cluster_dict=node_cluster_means,
    noncluster_dict=node_noncluster_means,
    title_prefix="Nodes (incl. walls) — ",
    save=False,
    file_suffix="_nodes_incl_walls",
    x_offset=0.0,
)

# B) Node features, EXCLUDING wall particles
plot_feature_list_compare(
    NODE_FEATURES,
    cluster_dict=node_cluster_means_nw,
    noncluster_dict=node_noncluster_means_nw,
    title_prefix="Nodes (no walls) — ",
    save=False,
    file_suffix="_nodes_no_walls",
    x_offset=0.0,
)

# C) Edge features (edges don't have "wall particles" to exclude here)
plot_feature_list_compare(
    EDGE_FEATURES,
    cluster_dict=edge_cluster_means,
    noncluster_dict=edge_noncluster_means,
    title_prefix="Edges — ",
    save=False,
    file_suffix="_edges",
    x_offset=0.0,
)

#### 3.1.7) Combined multi-panel figure (all features, cluster vs non-cluster)

In [ ]:
# ── Build "all" (pooled) stats needed for this plot ─────────────────────────
node_stats_mean_all    = {}
node_stats_mean_all_nw = {}
edge_stats_mean_all    = {}

for geom in geom_keys:
    all_node_rows    = node_stats[geom]["cluster"] + node_stats[geom]["noncluster"]
    all_node_rows_nw = filter_out_walls(all_node_rows)
    all_edge_rows    = edge_stats[geom]["cluster"] + edge_stats[geom]["noncluster"]

    node_stats_mean_all[geom]    = compute_mean_std(all_node_rows,    NODE_FEATURES)
    node_stats_mean_all_nw[geom] = compute_mean_std(all_node_rows_nw, NODE_FEATURES)
    edge_stats_mean_all[geom]    = compute_mean_std(all_edge_rows,    EDGE_FEATURES)


def plot_feature_list_all(
    feature_list,
    all_dict,
    title_prefix="",
    save=False,
    save_dir=None,
    file_suffix="",
    dpi=200,
):
    """
    One figure per feature showing mean ± stderr for ALL nodes
    (cluster + non-cluster pooled) across geometries.
    Colour = geometry.
    """
    if save:
        if save_dir is None:
            raise ValueError("save_dir must be set when save=True.")
        os.makedirs(save_dir, exist_ok=True)

    xs = np.arange(len(geom_keys))

    for feature in feature_list:
        fig, ax = plt.subplots(figsize=(6.5, 4.5))

        for i, g in enumerate(geom_keys):
            mean_a, err_a = all_dict[g][feature]
            ax.errorbar(
                xs[i], mean_a,
                yerr=err_a,
                fmt="o",
                markersize=8,
                capsize=3,
                color=palette[i],
                ecolor=palette[i],
                elinewidth=1.2,
                label=geom_labels[i],
            )

        ax.set_xticks(xs)
        ax.set_xticklabels(geom_labels)
        ax.set_ylabel(f"Mean {feature}")
        ax.set_title(f"{title_prefix}{feature}")
        ax.grid(False)

        plt.tight_layout()

        if save:
            fpath = os.path.join(save_dir, f"{feature}{file_suffix}.png")
            fig.savefig(fpath, dpi=dpi)
            print(f"Saved: {fpath}")

        plt.show()


# Node features (no-wall), all nodes pooled — varies only by geometry
plot_feature_list_all(
    NODE_FEATURES,
    all_dict=node_stats_mean_all_nw,
    title_prefix="Nodes (no walls, all) — ",
    save=False,
    file_suffix="_nodes_all_no_wall",
)

# Edge features, all edges pooled — varies only by geometry
plot_feature_list_all(
    EDGE_FEATURES,
    all_dict=edge_stats_mean_all,
    title_prefix="Edges (all) — ",
    save=False,
    file_suffix="_edges_all",
)


#### 3.1.8) Save feature stats as DataFrames (cluster / non-cluster / all)

In [ ]:
# node_stats_mean_all / node_stats_mean_all_nw / edge_stats_mean_all
# are already built in the cell above (3.1.7).

def _stats_to_df(cluster_dict, noncluster_dict, all_dict, features, feat_type):
    """
    Return a DataFrame with columns:
        geom | feature | group | mean | stderr | feature_type
    Groups: 'cluster', 'noncluster', 'all'
    """
    rows = []
    for geom in geom_keys:
        for feat in features:
            m_c,  s_c  = cluster_dict[geom][feat]
            m_nc, s_nc = noncluster_dict[geom][feat]
            m_a,  s_a  = all_dict[geom][feat]
            rows.append(dict(geom=geom, feature=feat, group="cluster",
                             mean=m_c,  stderr=s_c,  feature_type=feat_type))
            rows.append(dict(geom=geom, feature=feat, group="noncluster",
                             mean=m_nc, stderr=s_nc, feature_type=feat_type))
            rows.append(dict(geom=geom, feature=feat, group="all",
                             mean=m_a,  stderr=s_a,  feature_type=feat_type))
    return pd.DataFrame(rows)


# Node features — including walls
# node_feature_df = _stats_to_df(
#     node_cluster_means,
#     node_noncluster_means,
#     node_stats_mean_all,
#     NODE_FEATURES,
#     feat_type="node",
# )

# Node features — excluding walls
node_feature_df_nw = _stats_to_df(
    node_cluster_means_nw,
    node_noncluster_means_nw,
    node_stats_mean_all_nw,
    NODE_FEATURES,
    feat_type="node",
)

# Edge features
edge_feature_df = _stats_to_df(
    edge_cluster_means,
    edge_noncluster_means,
    edge_stats_mean_all,
    EDGE_FEATURES,
    feat_type="edge",
)

# Merge all into one table
combined_feature_df = pd.concat(
    [node_feature_df_nw, edge_feature_df],
    ignore_index=True,
)

print(combined_feature_df.head(12))
print(f"\nShape: {combined_feature_df.shape}")
print(f"Groups: {combined_feature_df['group'].unique()}")
print(f"Feature types: {combined_feature_df['feature_type'].unique()}")

# ── Optional: save to CSV ────────────────────────────────────
combined_feature_df.to_csv(os.path.join(out_dir, "feature_stats_cluster_comparison.csv"), index=False)
node_feature_df_nw.to_csv(os.path.join(out_dir, "node_feature_stats_no_wall.csv"), index=False)
edge_feature_df.to_csv(os.path.join(out_dir, "edge_feature_stats.csv"), index=False)


In [ ]:

# Degree histogram: 0° and 30°, all no-wall nodes pooled (MATLAB-style integer bins)
x_min, x_max = 2, 11
bin_edges = np.arange(x_min - 0.5, x_max + 1.5, 1)  # bins centred on 2, 3, ..., 11

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False)

for ax, geom, label, color in zip(
    axes,
    ["0deg", "30deg"],
    ["0°",   "30°"],
    [palette[0], palette[2]],
):
    all_rows = filter_out_walls(
        node_stats[geom]["cluster"] + node_stats[geom]["noncluster"]
    )
    vals = np.array([r["degree"] for r in all_rows], dtype=float)
    vals = vals[~np.isnan(vals)]

    ax.hist(vals, bins=bin_edges, density=True,
            color=color, alpha=0.6, edgecolor="white", linewidth=0.6)

    ax.set_xlim(x_min - 0.5, x_max + 0.5)
    ax.set_xticks(np.arange(x_min, x_max + 1, 1))
    ax.set_xlabel("Degree")
    ax.set_ylabel("Probability density")
    ax.set_title(f"Degree distribution — {label}")
    ax.grid(False)

plt.tight_layout()
plt.show()


### 3.2) Force cluster density comparison

#### 3.2.1) Cluster density calculation using average degree

In [ ]:
# --------------------------------------------------------
# 1) Collect densities for TYPE-1 clusters only
#    - cluster_density_type1_all  : avg degree over ALL nodes in Gc
#    - cluster_density_type1_core : avg degree over NON-WALL nodes,
#                                   but degrees are still from Gc
# --------------------------------------------------------

cluster_density_type1_all  = {g: [] for g in geom_keys}
cluster_density_type1_core = {g: [] for g in geom_keys}

for geom in geom_keys:
    sim_clusters_list = clusters_index[geom]  # list over simulations

    for clusters in sim_clusters_list:
        for rec in clusters:
            Gc    = rec["graph"]   # high-force cluster subgraph (includes walls)
            ctype = rec["type"]    # -100, 100, or 1

            # Only use type-1 clusters
            if ctype != 1:
                continue

            if Gc.number_of_nodes() == 0:
                continue

            # Degrees in the full cluster subgraph (walls included)
            deg_dict = dict(Gc.degree())

            # ---------- (A) Avg degree INCLUDING wall nodes ----------
            degs_all = list(deg_dict.values())
            if len(degs_all) > 0:
                avg_deg_all = float(np.mean(degs_all))
                cluster_density_type1_all[geom].append(avg_deg_all)

            # ---------- (B) Avg degree EXCLUDING wall nodes ----------
            # Still use degrees from Gc, but only average over non-wall nodes
            core_degs = []
            for n, data in Gc.nodes(data=True):
                if not data.get("is_wall", False):
                    core_degs.append(deg_dict[n])

            if len(core_degs) > 0:
                avg_deg_core = float(np.mean(core_degs))
                cluster_density_type1_core[geom].append(avg_deg_core)

# Quick sanity check
print("=== TYPE-1 clusters: ALL nodes (degrees from Gc) ===")
for geom in geom_keys:
    vals = np.array(cluster_density_type1_all[geom], float)
    vals = vals[~np.isnan(vals)]
    print(f"{geom}: {len(vals)} clusters, mean avg-degree = {np.mean(vals) if len(vals) > 0 else np.nan:.3f}")

print("\n=== TYPE-1 clusters: CORE nodes only (degrees from Gc) ===")
for geom in geom_keys:
    vals = np.array(cluster_density_type1_core[geom], float)
    vals = vals[~np.isnan(vals)]
    print(f"{geom}: {len(vals)} clusters, mean avg-degree = {np.mean(vals) if len(vals) > 0 else np.nan:.3f}")

#### 3.2.2) Plot cluster density distribution

In [ ]:
# Ensure these exist (as before)
geom_keys   = ["0deg", "15deg", "30deg", "45deg"]
geom_labels = ["0°",   "15°",   "30°",   "45°"]
palette     = ["#7B68EE", "#1E90FF", "#3CB371", "#FFD700"]

# Font style (same as other plots)
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 16
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 18
plt.rcParams["xtick.labelsize"] = 16
plt.rcParams["ytick.labelsize"] = 16

def plot_cluster_density_histogram(
    cluster_density,
    bins=20,
    title="Cluster density histogram",
    alpha=0.4,
    save=False,
    save_path=None,
    dpi=200
):
    """
    cluster_density: dict {geom_key: [avg_degree, ...]}
    bins: number of bins OR list of bin edges
    """

    # -----------------------------------------
    # 1) Combine all values to determine bin edges
    # -----------------------------------------
    all_vals = []
    for geom in geom_keys:
        vals = np.array(cluster_density[geom], dtype=float)
        vals = vals[~np.isnan(vals)]
        all_vals.extend(vals)
    
    all_vals = np.array(all_vals)
    if len(all_vals) == 0:
        print("No data to plot.")
        return
    
    # Compute shared bin edges
    bin_edges = np.histogram_bin_edges(all_vals, bins=bins)

    # -----------------------------------------
    # 2) Plot with shared bins
    # -----------------------------------------
    fig, ax = plt.subplots(figsize=(7, 4.5))

    for geom, color, label in zip(geom_keys, palette, geom_labels):
        vals = np.array(cluster_density[geom], dtype=float)
        vals = vals[~np.isnan(vals)]

        if len(vals) == 0:
            continue

        ax.hist(
            vals,
            bins=bin_edges,
            density=True,
            alpha=alpha,
            color=color,
            edgecolor="none",
            label=label,
        )

    # Labels & style
    ax.set_xlabel("Cluster average degree")
    ax.set_ylabel("Probability density")
    ax.set_title(title)
    ax.grid(False)
    ax.legend(frameon=False)

    plt.tight_layout()

    if save and save_path is not None:
        fig.savefig(save_path, dpi=dpi)
        print(f"Saved: {save_path}")

    plt.show()


# (1) Type-1 clusters: average degree over ALL nodes
plot_cluster_density_histogram(
    cluster_density_type1_all,
    bins=16,
    title="Type-1 Force-Cluster Density (All Nodes)"
)

# (2) Type-1 clusters: average degree over NON-WALL nodes (using same degrees)
plot_cluster_density_histogram(
    cluster_density_type1_core,
    bins=16,
    title="Type-1 Force-Cluster Density (Core Nodes Only)"
)

## 4) Loops in different geometries

### 4.1) Optional rough loop estimate

This cell is kept for quick experiments only. The production graph lists already carry loop metrics as graph attributes, and some folders also include `mcb_loop_sizes.pkl` / loop-stat CSVs. Prefer section 4.2 for normal analysis.

In [ ]:

geom_keys   = ["0deg", "15deg", "30deg", "45deg"]
geom_labels = ["0°",   "15°",   "30°",   "45°"]
palette     = ["#7B68EE", "#1E90FF", "#3CB371", "#FFD700"]

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 16
plt.rcParams["axes.titlesize"] = 18
plt.rcParams["axes.labelsize"] = 18
plt.rcParams["xtick.labelsize"] = 16
plt.rcParams["ytick.labelsize"] = 16

# -------------------------------------------------
# Loop detection strategy:
#
#   cycle_basis + chord check alone is NOT equivalent to minimum_cycle_basis.
#   A large "chordless" cycle (no direct shortcut among its own nodes) can
#   still be decomposed into small loops via paths through the rest of the
#   graph — which is what minimum_cycle_basis does.
#
#   However for SMALL cycles the approach is correct:
#     - Triangles (size 3): always irreducible.
#     - Quads (size 4): chordless quad = real independent 4-loop.
#     - Size 5-N: chord check still catches obvious composites.
#
#   The fix: keep both the chord check AND a hard MAX_SIZE cap.
#   Anything larger than MAX_SIZE from cycle_basis is unreliable and dropped.
# -------------------------------------------------

MAX_LOOP_SIZE = 10   # granular packings: physically meaningful loops are size 3–8

def small_chordless_cycle_sizes(G, max_size=MAX_LOOP_SIZE):
    H = nx.k_core(G, k=2)
    if H.number_of_edges() == 0:
        return []

    sizes = []
    for cyc in nx.cycle_basis(H):
        k = len(cyc)
        if k < 3 or k > max_size:
            continue
        if H.subgraph(cyc).number_of_edges() == k:   # chord check
            sizes.append(k)

    return sizes


# -------------------------------------------------
# Main computation
# -------------------------------------------------
loop_sizes             = {g: [] for g in geom_keys}
loop_counts_per_sim    = {g: [] for g in geom_keys}
loop_mean_size_per_sim = {g: [] for g in geom_keys}
loop_frac_tri_per_sim  = {g: [] for g in geom_keys}

for geom in geom_keys:
    print("Processing loops in geometry:", geom)
    for G in GL[geom]["full"]:
        sizes = small_chordless_cycle_sizes(G)
        loop_sizes[geom].extend(sizes)

        if len(sizes) == 0:
            loop_counts_per_sim[geom].append(0)
            loop_mean_size_per_sim[geom].append(np.nan)
            loop_frac_tri_per_sim[geom].append(np.nan)
        else:
            arr = np.array(sizes)
            loop_counts_per_sim[geom].append(len(arr))
            loop_mean_size_per_sim[geom].append(float(np.mean(arr)))
            loop_frac_tri_per_sim[geom].append(float(np.sum(arr == 3) / len(arr)))

# Summary
print(f"\n=== Loop stats (cycle_basis + chord check, size ≤ {MAX_LOOP_SIZE}) ===")
for geom in geom_keys:
    vals = np.array(loop_sizes[geom])
    if len(vals) == 0:
        print(f"{geom}: no loops found")
    else:
        print(
            f"{geom}: n_loops={len(vals)}, "
            f"mean={np.mean(vals):.2f}, median={np.median(vals):.2f}, "
            f"min={np.min(vals):.0f}, max={np.max(vals):.0f}"
        )

print(f"\n{'Geom':<8} {'n_sims':>7} {'mean_loops/sim':>16} {'mean_loop_size':>16} {'frac_triangles':>16}")
print("-" * 65)
for geom, label in zip(geom_keys, geom_labels):
    counts = np.array(loop_counts_per_sim[geom])
    msizes = np.array(loop_mean_size_per_sim[geom])
    ftri   = np.array(loop_frac_tri_per_sim[geom])
    print(
        f"{label:<8} {len(counts):>7} "
        f"{np.mean(counts):>16.1f} "
        f"{np.nanmean(msizes):>16.3f} "
        f"{np.nanmean(ftri)*100:>15.1f}%"
    )


### 4.2) Load loop metrics already saved with the graph list

The current graph structure stores loop metrics on each graph object (`G.graph`) and/or in `mcb_loop_sizes.pkl`, so the notebook should load them instead of recomputing minimum-cycle-basis loops.

In [ ]:
import pickle

geom_keys   = [g for g in ["0deg", "15deg", "30deg", "45deg"] if g in GL]
geom_labels = [g.replace("deg", "°") for g in geom_keys]
palette     = ["#7B68EE", "#1E90FF", "#3CB371", "#FFD700"][:len(geom_keys)]

LOOP_GRAPH_VIEW = "core"  # "core" = no wall nodes; "full" = with wall nodes
suffix = "_with_walls" if LOOP_GRAPH_VIEW == "full" else ""
loop_sizes_key = f"loop_sizes{suffix}"
loop_counts_key = f"loop_counts{suffix}"
loop_total_key = f"loop_total{suffix}"
loop_mean_key = f"loop_mean{suffix}"


def _expand_saved_loop_sizes(graph_attrs, sizes_key, counts_key):
    sizes = graph_attrs.get(sizes_key)
    if sizes is not None:
        return [int(s) for s in sizes]

    counts = graph_attrs.get(counts_key, {}) or {}
    expanded = []
    for size, count in sorted(counts.items(), key=lambda kv: int(kv[0])):
        expanded.extend([int(size)] * int(count))
    return expanded


def _load_loop_metrics_from_graph_attrs(GL_dict):
    saved_loop_sizes             = {g: [] for g in geom_keys}
    saved_loop_counts_per_sim    = {g: [] for g in geom_keys}
    saved_loop_mean_size_per_sim = {g: [] for g in geom_keys}
    saved_loop_frac_tri_per_sim  = {g: [] for g in geom_keys}

    found_any = False
    for geom in geom_keys:
        for G in GL_dict[geom][LOOP_GRAPH_VIEW]:
            arr = np.asarray(_expand_saved_loop_sizes(G.graph, loop_sizes_key, loop_counts_key), dtype=int)
            if arr.size or loop_total_key in G.graph or loop_counts_key in G.graph:
                found_any = True
            saved_loop_sizes[geom].extend(arr.tolist())

            if arr.size:
                saved_loop_counts_per_sim[geom].append(int(G.graph.get(loop_total_key, len(arr))))
                saved_loop_mean_size_per_sim[geom].append(float(G.graph.get(loop_mean_key, np.mean(arr))))
                saved_loop_frac_tri_per_sim[geom].append(float(np.sum(arr == 3) / len(arr)))
            else:
                saved_loop_counts_per_sim[geom].append(int(G.graph.get(loop_total_key, 0)))
                saved_loop_mean_size_per_sim[geom].append(float(G.graph.get(loop_mean_key, np.nan)))
                saved_loop_frac_tri_per_sim[geom].append(np.nan)

    if not found_any:
        raise KeyError("No saved loop attributes found on graph objects.")
    return saved_loop_sizes, saved_loop_counts_per_sim, saved_loop_mean_size_per_sim, saved_loop_frac_tri_per_sim


mcb_cache = GRAPH_DIR / "mcb_loop_sizes.pkl"
try:
    loop_sizes, loop_counts_per_sim, loop_mean_size_per_sim, loop_frac_tri_per_sim = _load_loop_metrics_from_graph_attrs(GL)
    print("Loaded loop metrics from graph attributes in graph_dict.")
except KeyError:
    if not mcb_cache.exists():
        raise
    with open(mcb_cache, "rb") as f:
        saved = pickle.load(f)
    loop_sizes             = saved["loop_sizes"]
    loop_counts_per_sim    = saved["counts_per_sim"]
    loop_mean_size_per_sim = saved["mean_size_per_sim"]
    loop_frac_tri_per_sim  = saved["frac_tri_per_sim"]
    print(f"Loaded loop metrics from cache: {mcb_cache}")

print("
=== Saved loop metrics ===")
for geom in geom_keys:
    vals = np.asarray(loop_sizes[geom])
    if vals.size == 0:
        print(f"{geom}: no loops found")
    else:
        print(
            f"{geom}: n_loops={len(vals)}, "
            f"mean_size={np.mean(vals):.2f}, "
            f"triangles={np.sum(vals == 3)}"
        )

### 4.3) Visualization and saving the loop stats

In [ ]:

# ================================================================
# Loop analysis: plots + save stats
# ================================================================

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"]        = 16
plt.rcParams["axes.titlesize"]   = 18
plt.rcParams["axes.labelsize"]   = 18
plt.rcParams["xtick.labelsize"]  = 16
plt.rcParams["ytick.labelsize"]  = 16

LOOP_PLOT_DIR = os.path.join(out_dir, "Plots", "Loops")
os.makedirs(LOOP_PLOT_DIR, exist_ok=True)

xs_geom = np.arange(len(geom_keys))

# ── 1) Loop-size frequency distribution ──────────────────────────────────────
# Pooled over all sims per geometry; bar plot centred on integer sizes 3..10

size_range = range(3, 11)
fig, ax = plt.subplots(figsize=(8, 4.5))
bar_w = 0.18
offsets = np.linspace(-(len(geom_keys)-1)/2, (len(geom_keys)-1)/2, len(geom_keys)) * bar_w

for i, (geom, label, color) in enumerate(zip(geom_keys, geom_labels, palette)):
    vals = np.array(loop_sizes[geom])
    total = len(vals) if len(vals) > 0 else 1
    counts = [np.sum(vals == s) / total for s in size_range]
    ax.bar(np.array(list(size_range)) + offsets[i], counts,
           width=bar_w, color=color, alpha=0.85, label=label)

ax.set_xlabel("Loop size (nodes)")
ax.set_ylabel("Fraction of loops")
ax.set_title("Loop-size distribution by geometry")
ax.set_xticks(list(size_range))
ax.legend(frameon=False, fontsize=14)
ax.grid(False)
plt.tight_layout()
fig.savefig(os.path.join(LOOP_PLOT_DIR, "loop_size_distribution.png"), dpi=200)
plt.show()

# ── 2) Mean loops per simulation ─────────────────────────────────────────────
counts_mean = [np.mean(loop_counts_per_sim[g]) for g in geom_keys]
counts_err  = [np.std(loop_counts_per_sim[g], ddof=1) / np.sqrt(len(loop_counts_per_sim[g]))
               for g in geom_keys]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for i, (m, e, color) in enumerate(zip(counts_mean, counts_err, palette)):
    ax.errorbar(xs_geom[i], m, yerr=e, fmt="o", markersize=9,
                capsize=4, color=color, ecolor=color, elinewidth=1.5)
ax.set_xticks(xs_geom)
ax.set_xticklabels(geom_labels)
ax.set_ylabel("Mean loops per simulation")
ax.set_title("Loop count per simulation")
ax.grid(False)
plt.tight_layout()
fig.savefig(os.path.join(LOOP_PLOT_DIR, "loop_count_per_sim.png"), dpi=200)
plt.show()

# ── 3) Mean loop size across geometries ──────────────────────────────────────
msize_mean = [np.nanmean(loop_mean_size_per_sim[g]) for g in geom_keys]
msize_err  = [np.nanstd(loop_mean_size_per_sim[g], ddof=1) /
              np.sqrt(np.sum(~np.isnan(loop_mean_size_per_sim[g])))
              for g in geom_keys]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for i, (m, e, color) in enumerate(zip(msize_mean, msize_err, palette)):
    ax.errorbar(xs_geom[i], m, yerr=e, fmt="s", markersize=9,
                capsize=4, color=color, ecolor=color, elinewidth=1.5)
ax.set_xticks(xs_geom)
ax.set_xticklabels(geom_labels)
ax.set_ylabel("Mean loop size")
ax.set_title("Mean loop size by geometry")
ax.grid(False)
plt.tight_layout()
fig.savefig(os.path.join(LOOP_PLOT_DIR, "mean_loop_size.png"), dpi=200)
plt.show()

# ── 4) Fraction of triangles ─────────────────────────────────────────────────
ftri_mean = [np.nanmean(loop_frac_tri_per_sim[g]) * 100 for g in geom_keys]
ftri_err  = [np.nanstd(loop_frac_tri_per_sim[g], ddof=1) /
             np.sqrt(np.sum(~np.isnan(loop_frac_tri_per_sim[g]))) * 100
             for g in geom_keys]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for i, (m, e, color) in enumerate(zip(ftri_mean, ftri_err, palette)):
    ax.errorbar(xs_geom[i], m, yerr=e, fmt="^", markersize=9,
                capsize=4, color=color, ecolor=color, elinewidth=1.5)
ax.set_xticks(xs_geom)
ax.set_xticklabels(geom_labels)
ax.set_ylabel("Triangle fraction (%)")
ax.set_title("Fraction of size-3 loops by geometry")
ax.grid(False)
plt.tight_layout()
fig.savefig(os.path.join(LOOP_PLOT_DIR, "frac_triangles.png"), dpi=200)
plt.show()

# ── 5) Save per-sim stats as CSV ─────────────────────────────────────────────
rows = []
for geom, label in zip(geom_keys, geom_labels):
    for sim_idx, (cnt, ms, ft) in enumerate(zip(
        loop_counts_per_sim[geom],
        loop_mean_size_per_sim[geom],
        loop_frac_tri_per_sim[geom],
    )):
        rows.append(dict(
            geom=geom, geom_label=label, sim_idx=sim_idx,
            n_loops=cnt, mean_loop_size=ms, frac_triangles=ft,
        ))

loop_stats_df = pd.DataFrame(rows)
csv_path = os.path.join(out_dir, "loop_stats_per_sim.csv")
loop_stats_df.to_csv(csv_path, index=False)
print(f"Saved per-sim loop stats → {csv_path}")
print(loop_stats_df.groupby("geom")[["n_loops", "mean_loop_size", "frac_triangles"]].mean().round(3))


### 4.4) Visualization directly from saved loop attributes in the clustered graph pickle

In [ ]:

# ================================================================
# Loop analysis from saved graph attributes
# ================================================================

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"]        = 16
plt.rcParams["axes.titlesize"]   = 18
plt.rcParams["axes.labelsize"]   = 18
plt.rcParams["xtick.labelsize"]  = 16
plt.rcParams["ytick.labelsize"]  = 16

LOOP_GRAPH_PICKLE = str(GRAPH_WITH_CLUSTERS_PATH if GRAPH_WITH_CLUSTERS_PATH.exists() else GRAPH_LABELED_PATH)
LOOP_GRAPH_VIEW = "core"  # use "core" for loop stats without wall nodes
LOOP_SAVE_TAG = "saved_attrs"

with open(LOOP_GRAPH_PICKLE, "rb") as f:
    GL_loops_saved = pickle.load(f)
print(f"Loaded saved graph data for loop attributes → {LOOP_GRAPH_PICKLE}")

suffix = "_with_walls" if LOOP_GRAPH_VIEW == "full" else ""
loop_sizes_key = f"loop_sizes{suffix}"
loop_counts_key = f"loop_counts{suffix}"
loop_total_key = f"loop_total{suffix}"
loop_mean_key = f"loop_mean{suffix}"

def _expand_saved_loop_sizes(graph_attrs, sizes_key, counts_key):
    sizes = graph_attrs.get(sizes_key)
    if sizes is not None:
        return [int(s) for s in sizes]

    counts = graph_attrs.get(counts_key, {}) or {}
    expanded = []
    for size, count in sorted(counts.items(), key=lambda kv: int(kv[0])):
        expanded.extend([int(size)] * int(count))
    return expanded

saved_loop_sizes             = {g: [] for g in geom_keys}
saved_loop_counts_per_sim    = {g: [] for g in geom_keys}
saved_loop_mean_size_per_sim = {g: [] for g in geom_keys}
saved_loop_frac_tri_per_sim  = {g: [] for g in geom_keys}

for geom in geom_keys:
    for G in GL_loops_saved[geom][LOOP_GRAPH_VIEW]:
        arr = np.asarray(_expand_saved_loop_sizes(G.graph, loop_sizes_key, loop_counts_key), dtype=int)
        saved_loop_sizes[geom].extend(arr.tolist())

        if arr.size:
            saved_loop_counts_per_sim[geom].append(int(G.graph.get(loop_total_key, len(arr))))
            saved_loop_mean_size_per_sim[geom].append(float(G.graph.get(loop_mean_key, np.mean(arr))))
            saved_loop_frac_tri_per_sim[geom].append(float(np.sum(arr == 3) / len(arr)))
        else:
            saved_loop_counts_per_sim[geom].append(int(G.graph.get(loop_total_key, 0)))
            saved_loop_mean_size_per_sim[geom].append(float(G.graph.get(loop_mean_key, np.nan)))
            saved_loop_frac_tri_per_sim[geom].append(np.nan)

LOOP_PLOT_DIR = os.path.join(out_dir, "Plots", "Loops")
os.makedirs(LOOP_PLOT_DIR, exist_ok=True)

xs_geom = np.arange(len(geom_keys))
bar_w = 0.18
offsets = np.linspace(-(len(geom_keys)-1)/2, (len(geom_keys)-1)/2, len(geom_keys)) * bar_w

def _saved_fraction_by_size(vals, size_values):
    vals = np.asarray(vals, dtype=int)
    total = len(vals) if len(vals) > 0 else 1
    return [np.sum(vals == s) / total for s in size_values]

# ── 1) Loop-size frequency distribution ──────────────────────────────────────
size_range = list(range(3, 11))
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, (geom, label, color) in enumerate(zip(geom_keys, geom_labels, palette)):
    counts = _saved_fraction_by_size(saved_loop_sizes[geom], size_range)
    ax.bar(np.array(size_range) + offsets[i], counts,
           width=bar_w, color=color, alpha=0.85, label=label)

ax.set_xlabel("Loop size (nodes)")
ax.set_ylabel("Fraction of loops")
ax.set_title("Loop-size distribution by geometry")
ax.set_xticks(size_range)
ax.legend(frameon=False, fontsize=14)
ax.grid(False)
plt.tight_layout()
fig.savefig(os.path.join(LOOP_PLOT_DIR, f"loop_size_distribution_{LOOP_SAVE_TAG}.png"), dpi=200)
plt.show()

# ── 1b) Zoomed loop-size frequency distribution (sizes 5..10) ───────────────
size_range_zoom = list(range(5, 11))
fig, ax = plt.subplots(figsize=(8, 4.5))
zoom_max = 0.0

for i, (geom, label, color) in enumerate(zip(geom_keys, geom_labels, palette)):
    counts = _saved_fraction_by_size(saved_loop_sizes[geom], size_range_zoom)
    zoom_max = max(zoom_max, max(counts))
    ax.bar(np.array(size_range_zoom) + offsets[i], counts,
           width=bar_w, color=color, alpha=0.85, label=label)

ax.set_xlabel("Loop size (nodes)")
ax.set_ylabel("Fraction of loops")
ax.set_title("Loop-size distribution by geometry (zoom: 5-10)")
ax.set_xticks(size_range_zoom)
ax.set_ylim(0, max(zoom_max * 1.15, 0.002))
ax.legend(frameon=False, fontsize=14)
ax.grid(False)
plt.tight_layout()
fig.savefig(os.path.join(LOOP_PLOT_DIR, f"loop_size_distribution_zoom_5_10_{LOOP_SAVE_TAG}.png"), dpi=200)
plt.show()

# ── 2) Mean loops per simulation ─────────────────────────────────────────────
counts_mean = [np.mean(saved_loop_counts_per_sim[g]) for g in geom_keys]
counts_err  = [np.std(saved_loop_counts_per_sim[g], ddof=1) / np.sqrt(len(saved_loop_counts_per_sim[g]))
               for g in geom_keys]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for i, (m, e, color) in enumerate(zip(counts_mean, counts_err, palette)):
    ax.errorbar(xs_geom[i], m, yerr=e, fmt="o", markersize=9,
                capsize=4, color=color, ecolor=color, elinewidth=1.5)
ax.set_xticks(xs_geom)
ax.set_xticklabels(geom_labels)
ax.set_ylabel("Mean loops per simulation")
ax.set_title("Loop count per simulation")
ax.grid(False)
plt.tight_layout()
fig.savefig(os.path.join(LOOP_PLOT_DIR, f"loop_count_per_sim_{LOOP_SAVE_TAG}.png"), dpi=200)
plt.show()

# ── 3) Mean loop size across geometries ──────────────────────────────────────
msize_mean = [np.nanmean(saved_loop_mean_size_per_sim[g]) for g in geom_keys]
msize_err  = [np.nanstd(saved_loop_mean_size_per_sim[g], ddof=1) /
              np.sqrt(np.sum(~np.isnan(saved_loop_mean_size_per_sim[g])))
              for g in geom_keys]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for i, (m, e, color) in enumerate(zip(msize_mean, msize_err, palette)):
    ax.errorbar(xs_geom[i], m, yerr=e, fmt="s", markersize=9,
                capsize=4, color=color, ecolor=color, elinewidth=1.5)
ax.set_xticks(xs_geom)
ax.set_xticklabels(geom_labels)
ax.set_ylabel("Mean loop size")
ax.set_title("Mean loop size by geometry")
ax.grid(False)
plt.tight_layout()
fig.savefig(os.path.join(LOOP_PLOT_DIR, f"mean_loop_size_{LOOP_SAVE_TAG}.png"), dpi=200)
plt.show()

# ── 4) Fraction of triangles ─────────────────────────────────────────────────
ftri_mean = [np.nanmean(saved_loop_frac_tri_per_sim[g]) * 100 for g in geom_keys]
ftri_err  = [np.nanstd(saved_loop_frac_tri_per_sim[g], ddof=1) /
             np.sqrt(np.sum(~np.isnan(saved_loop_frac_tri_per_sim[g]))) * 100
             for g in geom_keys]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for i, (m, e, color) in enumerate(zip(ftri_mean, ftri_err, palette)):
    ax.errorbar(xs_geom[i], m, yerr=e, fmt="^", markersize=9,
                capsize=4, color=color, ecolor=color, elinewidth=1.5)
ax.set_xticks(xs_geom)
ax.set_xticklabels(geom_labels)
ax.set_ylabel("Triangle fraction (%)")
ax.set_title("Fraction of size-3 loops by geometry")
ax.grid(False)
plt.tight_layout()
fig.savefig(os.path.join(LOOP_PLOT_DIR, f"frac_triangles_{LOOP_SAVE_TAG}.png"), dpi=200)
plt.show()

# ── 5) Save per-sim stats as CSV ─────────────────────────────────────────────
rows = []
for geom, label in zip(geom_keys, geom_labels):
    for sim_idx, (cnt, ms, ft) in enumerate(zip(
        saved_loop_counts_per_sim[geom],
        saved_loop_mean_size_per_sim[geom],
        saved_loop_frac_tri_per_sim[geom],
    )):
        rows.append(dict(
            geom=geom, geom_label=label, sim_idx=sim_idx,
            n_loops=cnt, mean_loop_size=ms, frac_triangles=ft,
        ))

saved_loop_stats_df = pd.DataFrame(rows)
csv_path = os.path.join(out_dir, f"loop_stats_per_sim_{LOOP_SAVE_TAG}.csv")
saved_loop_stats_df.to_csv(csv_path, index=False)
print(f"Saved per-sim loop stats → {csv_path}")
print(saved_loop_stats_df.groupby("geom")[["n_loops", "mean_loop_size", "frac_triangles"]].mean().round(3))


# 3D visualizations of granular systems

## 3-D visualisation – random simulation per geometry, coloured by hydrostatic stress

Parameters for degree

- GRAPH_VIEW  = 'core'          # 'core' (particles only) | 'full' (+ wall cubes)
- NODE_ATTR   = 'degree'        # node attribute for colour
- NODE_SCALE  = 1               # 1 → higher degree = brighter; use -1 to invert
- EDGE_ATTR   = None            # keep edges gray; set e.g. 'normal_force' to colour
- CMAP        = 'parula'        # colormap
- ELEV, AZIM  = 25, 60         # 3-D viewing angle

- SPHERE_NODES     = False  # True → 3-D sphere meshes; False → flat scatter markers
- NODE_RESOLUTION  = 12     # sphere segments (8=fast, 12-16=publication quality)
- SPHERE_SCALE     = 1.0    # radius multiplier; uses node 'radius' attr if present,
                           # else half the median contact-edge length
- MARKER_SIZE      = 60    # scatter marker size in pts² (ignored when SPHERE_NODES=True)

- EDGE_WIDTH   = 0.75        # line width for edges

- ALPHA_MODE       = 'radial'   # 'radial' | 'attr' | 'none'
- ALPHA_ATTR       = 'degree'   # node attribute that drives opacity
- ALPHA_ATTR_SCALE = 1        # 1 → higher degree = more opaque
- ALPHA_MAX     = 1.0    # opacity at high end of scaled attribute
- ALPHA_MIN     = 0.75    # opacity at low end
- ALPHA_UNIFORM = 0.85   # used only when ALPHA_MODE = 'none'
- EDGE_ALPHA    = 0.4    # fixed edge opacity; None → mean of endpoint node opacities

- BG        = 'white'
- GRID      = False
- FIGSIZE   = (6, 6)

- NODE_VMIN, NODE_VMAX = 3, 10

In [ ]:
import sys, os, importlib
sys.path.insert(0, str(POST_ANALYSIS_DIR))
import graph3d_viz
importlib.reload(graph3d_viz)
from graph3d_viz import plot_graph_3d

# ── Settings ────────────────────────────────────────────────────────────────
GRAPH_VIEW  = 'core'          # 'core' (particles only) | 'full' (+ wall cubes)
NODE_ATTR   = 'degree'        # node attribute for colour
NODE_SCALE  = 1               # 1 → higher degree = brighter; use -1 to invert
EDGE_ATTR   = None            # keep edges gray; set e.g. 'normal_force' to colour
CMAP        = 'parula'        # colormap
ELEV, AZIM  = 25, 60         # 3-D viewing angle

# ── Node rendering ────────────────────────────────────────────────────────────
SPHERE_NODES     = False  # True → 3-D sphere meshes; False → flat scatter markers
NODE_RESOLUTION  = 12     # sphere segments (8=fast, 12-16=publication quality)
SPHERE_SCALE     = 1.0    # radius multiplier; uses node 'radius' attr if present,
                           # else half the median contact-edge length
MARKER_SIZE      = 60    # scatter marker size in pts² (ignored when SPHERE_NODES=True)

# ── Edge rendering ────────────────────────────────────────────────────────────
EDGE_WIDTH   = 0.75        # line width for edges

# ── Transparency ─────────────────────────────────────────────────────────────
ALPHA_MODE       = 'radial'   # 'radial' | 'attr' | 'none'
ALPHA_ATTR       = 'degree'   # node attribute that drives opacity
ALPHA_ATTR_SCALE = 1        # 1 → higher degree = more opaque
ALPHA_MAX     = 1.0    # opacity at high end of scaled attribute
ALPHA_MIN     = 0.75    # opacity at low end
ALPHA_UNIFORM = 0.85   # used only when ALPHA_MODE = 'none'
EDGE_ALPHA    = 0.4    # fixed edge opacity; None → mean of endpoint node opacities

# ── Figure / background ───────────────────────────────────────────────────────
BG        = 'white'
GRID      = False
FIGSIZE   = (6, 6)

# ── Color range (None → auto-shared across all panels) ───────────────────────
NODE_VMIN, NODE_VMAX = 3, 10

# ── Save ──────────────────────────────────────────────────────────────────────
SAVE_DIR = str(GRAPH_DIR / 'Plots')

# ── Plot ─────────────────────────────────────────────────────────────────────
figs = plot_graph_3d(
    GL_final,
    geometries       = None,
    sim_indices      = None,
    graph_view       = GRAPH_VIEW,
    node_attr        = NODE_ATTR,
    node_attr_scale  = NODE_SCALE,
    edge_attr        = EDGE_ATTR,
    edge_alpha       = EDGE_ALPHA,
    edge_width       = EDGE_WIDTH,
    cmap             = CMAP,
    node_vmin        = NODE_VMIN,
    node_vmax        = NODE_VMAX,
    elev             = ELEV,
    azim             = AZIM,
    bg               = BG,
    figsize          = FIGSIZE,
    separate_figures = True,
    grid             = GRID,
    alpha_mode       = ALPHA_MODE,
    alpha_attr       = ALPHA_ATTR,
    alpha_attr_scale = ALPHA_ATTR_SCALE,
    alpha_max        = ALPHA_MAX,
    alpha_min        = ALPHA_MIN,
    alpha_uniform    = ALPHA_UNIFORM,
    sphere_nodes     = SPHERE_NODES,
    node_resolution  = NODE_RESOLUTION,
    sphere_scale     = SPHERE_SCALE,
    marker_size      = MARKER_SIZE,
    save_dir         = SAVE_DIR,
    fmt              = 'png',
    dpi              = 300,
    show             = True,
)
print(f"✅ Plotted {len(figs)} figures.")


## 3D visualization of high force clusters

In [ ]:

# ----- Simple styling ----- 
EDGE_GRAY = mcolors.to_rgba("#bdbdbd")
NODE_GRAY = mcolors.to_rgba("#7f7f7f")

EDGE_W_OTHER = 0.75
EDGE_W_HF    = 1.5
ALPHA_OTHER  = 0.5
ALPHA_HF     = 0.90

NODE_SIZE_PART = 20
NODE_SIZE_WALL = 26
NODE_ALPHA_HF  = 0.95
NODE_ALPHA_OTHER = 0.5

def _pos(G, n):
    """Particle: node['position']; Wall: avg of incident 'contact_location'."""
    if not G.nodes[n].get("is_wall", False):
        x, y, z = G.nodes[n]["position"]
        return float(x), float(y), float(z)
    locs = []
    if G.is_multigraph():
        for _, _, _, d in G.edges(n, keys=True, data=True):
            if "contact_location" in d:
                locs.append(d["contact_location"])
    else:
        for _, _, d in G.edges(n, data=True):
            if "contact_location" in d:
                locs.append(d["contact_location"])
    if locs:
        xs, ys, zs = zip(*locs)
        return (sum(xs)/len(xs), sum(ys)/len(ys), sum(zs)/len(zs))
    return (0.0, 0.0, 0.0)

def _equal_axes_3d(ax, xs, ys, zs):
    xr = max(xs) - min(xs); yr = max(ys) - min(ys); zr = max(zs) - min(zs)
    R = max(xr, yr, zr) or 1.0
    cx = 0.5*(max(xs)+min(xs)); cy = 0.5*(max(ys)+min(ys)); cz = 0.5*(max(zs)+min(zs))
    ax.set_xlim(cx - R/2, cx + R/2)
    ax.set_ylim(cy - R/2, cy + R/2)
    ax.set_zlim(cz - R/2, cz + R/2)

def _cluster_color(cid):
    """Deterministic color per cluster id for non-(-100) clusters."""
    cmap = cm.get_cmap("tab20")
    return mcolors.to_rgba(cmap(((cid - 1) % 20) / 19.0))

def plot_hf_by_cluster_3d(G, title=None, elev=25, azim=60, n_ticks=3):
    """
    Plot FULL graph in 3D:
      - Non-HF edges: thin transparent gray
      - HF edges type -100: thick gray
      - Other HF edges: colored by hf_cluster_id
      - Nodes take the color of their cluster's edges (HF); others gray
      - Wall nodes '^', particles 'o'

    Parameters
    ----------
    G       : networkx Graph (full, labeled with hf_cluster_id / hf_cluster_type)
    title   : str, optional
    elev    : float  vertical viewing angle in degrees (default 25)
    azim    : float  horizontal viewing angle in degrees (default 60)
    n_ticks : int    number of ticks on each axis (default 3)
    """
    fig = plt.figure(figsize=(6.5, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.view_init(elev=elev, azim=azim)

    # Bounds
    xs, ys, zs = [], [], []
    for n in G.nodes:
        x, y, z = _pos(G, n); xs.append(x); ys.append(y); zs.append(z)

    # Draw non-HF edges
    if G.is_multigraph():
        for u, v, k, d in G.edges(keys=True, data=True):
            if d.get("is_high_force", False): continue
            x1,y1,z1 = _pos(G,u); x2,y2,z2 = _pos(G,v)
            ax.plot([x1,x2],[y1,y2],[z1,z2], color=EDGE_GRAY, lw=EDGE_W_OTHER, alpha=ALPHA_OTHER)
    else:
        for u, v, d in G.edges(data=True):
            if d.get("is_high_force", False): continue
            x1,y1,z1 = _pos(G,u); x2,y2,z2 = _pos(G,v)
            ax.plot([x1,x2],[y1,y2],[z1,z2], color=EDGE_GRAY, lw=EDGE_W_OTHER, alpha=ALPHA_OTHER)

    # Draw HF edges colored by cluster id; track node colors
    node_color = {}

    def _paint_edge(u, v, color, lw, alpha):
        x1,y1,z1 = _pos(G,u); x2,y2,z2 = _pos(G,v)
        ax.plot([x1,x2],[y1,y2],[z1,z2], color=color, lw=lw, alpha=alpha)

    if G.is_multigraph():
        for u, v, k, d in G.edges(keys=True, data=True):
            if not d.get("is_high_force", False): continue
            ctype = d.get("hf_cluster_type", 1)
            color = EDGE_GRAY if ctype == -100 else _cluster_color(d.get("hf_cluster_id", 1))
            _paint_edge(u, v, color, EDGE_W_HF, ALPHA_HF)
            node_color[u] = color; node_color[v] = color
    else:
        for u, v, d in G.edges(data=True):
            if not d.get("is_high_force", False): continue
            ctype = d.get("hf_cluster_type", 1)
            color = EDGE_GRAY if ctype == -100 else _cluster_color(d.get("hf_cluster_id", 1))
            _paint_edge(u, v, color, EDGE_W_HF, ALPHA_HF)
            node_color[u] = color; node_color[v] = color

    # Draw nodes
    def _scatter(nodes, marker, colors, alpha, size):
        if not nodes: return
        XYZ = [_pos(G, n) for n in nodes]
        X, Y, Z = zip(*XYZ)
        ax.scatter(X, Y, Z, marker=marker, s=size, c=colors, alpha=alpha, edgecolors="none")

    wall = [n for n in G.nodes if G.nodes[n].get("is_wall", False)]
    part = [n for n in G.nodes if not G.nodes[n].get("is_wall", False)]

    _scatter([n for n in part if n not in node_color], "o", [NODE_GRAY]*sum(1 for n in part if n not in node_color), NODE_ALPHA_OTHER, NODE_SIZE_PART)
    _scatter([n for n in wall if n not in node_color], "^", [NODE_GRAY]*sum(1 for n in wall if n not in node_color), NODE_ALPHA_OTHER, NODE_SIZE_WALL)
    _scatter([n for n in part if n in node_color],    "o", [node_color[n] for n in part if n in node_color], NODE_ALPHA_HF, NODE_SIZE_PART)
    _scatter([n for n in wall if n in node_color],    "^", [node_color[n] for n in wall if n in node_color], NODE_ALPHA_HF, NODE_SIZE_WALL)

    _equal_axes_3d(ax, xs, ys, zs)

    # Reduce tick density
    ax.xaxis.set_major_locator(plt.MaxNLocator(n_ticks))
    ax.yaxis.set_major_locator(plt.MaxNLocator(n_ticks))
    ax.zaxis.set_major_locator(plt.MaxNLocator(n_ticks))

    ax.grid(False)
    ax.set_xlabel(""); ax.set_ylabel(""); ax.set_zlabel("")
    if title: ax.set_title(title, pad=6)
    plt.tight_layout()
    return fig, ax


# ── Quick test: one random sim per geometry ───────────────────────────────────
def plot_random_per_geometry_by_id(GL_dict, seed=0, elev=25, azim=60, n_ticks=3):
    rng = random.Random(seed)
    figs = {}
    for angle, dd in GL_dict.items():
        if not dd["full"]: continue
        idx = rng.randrange(len(dd["full"]))
        G = dd["full"][idx]
        fig, ax = plot_hf_by_cluster_3d(G, title=f"{angle} — sim {idx}", elev=elev, azim=azim, n_ticks=n_ticks)
        figs[angle] = (idx, fig, ax)
    return figs


In [ ]:

# ================================================================
# Batch plot: HF clusters for ALL simulations in GL
# ================================================================

# ── Settings ─────────────────────────────────────────────────────
ELEV   = 20          # vertical viewing angle (degrees)
AZIM   = 30          # horizontal viewing angle (degrees)

SAVE   = True        # False → just display; True → save to disk
SHOW   = False       # True → also show inline (slow for many graphs)
DPI    = 200

HF_PLOT_DIR = str(GRAPH_DIR / "Plots" / "HF_Clusters")

# ── Which geometries / sims to plot ─────────────────────────────
GEOMS_TO_PLOT = geom_keys          # or e.g. ["0deg", "30deg"]
SIM_INDICES   = [0, 4, 7, 10, 15]               # None → all sims; or e.g. [0, 1, 2]

# ── Run ──────────────────────────────────────────────────────────
if SAVE:
    os.makedirs(HF_PLOT_DIR, exist_ok=True)

for geom in GEOMS_TO_PLOT:
    graphs = GL[geom]["full"]
    indices = SIM_INDICES if SIM_INDICES is not None else range(len(graphs))

    for sim_idx in indices:
        G = graphs[sim_idx]
        label = f"{geom}_sim{sim_idx:02d}"
        fig, ax = plot_hf_by_cluster_3d(
            G,
            title=f"{geom}  sim {sim_idx}  (elev={ELEV}°, azim={AZIM}°)",
            elev=ELEV,
            azim=AZIM,
        )

        if SAVE:
            fpath = os.path.join(HF_PLOT_DIR, f"hf_clusters_{label}_e{ELEV}_a{AZIM}.png")
            fig.savefig(fpath, dpi=DPI, bbox_inches="tight")
            print(f"Saved: {fpath}")

        if SHOW:
            plt.show()
        else:
            plt.close(fig)

print("✅ Done.")
